In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append("..")

import json
import pickle
import logging
import os

import pandas as pd
import numpy as np

from itext2kg import iText2KG
from itext2kg.utils import PubtatorProcessor
from langchain_ollama import ChatOllama, OllamaEmbeddings
from itext2kg.graph_integration import GraphIntegrator

/home/mindrank/miniconda3/envs/kg/lib/python3.11/site-packages/pydantic/_internal/_generate_schema.py:502: UserWarning: <built-in function array> is not a Python type (it may be an instance of an object), Pydantic will allow any object with no validation since we cannot even enforce that the input is an instance of the given type. To get rid of this error wrap the type with `pydantic.SkipValidation`.
  warn(


In [2]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

pmid= 10869059

In [3]:
NEO4J_URI = "neo4j+s://435eb4f0.databases.neo4j.io"   
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = "nqL3At3rptFpfRrBhg5Y1-7k78oHqVaW42WKtqVRnJo"

# llm = ChatOllama(
#     model="deepseek-r1:32b",
#     temperature=0,
# )

llm = True

embeddings = OllamaEmbeddings(
    model="nomic-embed-text:latest",
)

In [4]:
DATA_PATH = "/home/mindrank/fuli/itext2kg/Data/delirium"
OUTPUT_PATH = "/home/mindrank/fuli/itext2kg/output_kg/delirium"
pmid_list =[]
for file_name in os.listdir(DATA_PATH):
    pmid = file_name.split('.')[0]
    if os.path.exists(f"{OUTPUT_PATH}/{pmid}.pkl"):
        pass
    else:
        pmid_list.append(pmid)

pmid_list_chunk = np.array_split(pmid_list, 10)

In [5]:
# for i in pmid_list_chunk[0]:
#     print(i)
#     break

In [6]:
PUTATOR_PATH = "/home/mindrank/fuli/itext2kg/Data/delirium"
JSON_PATH = "/home/mindrank/fuli/itext2kg/output_kg/"
PMID = 36824441

In [7]:
with open(f"{PUTATOR_PATH}/{PMID}.txt", "r") as f:
    text = f.readlines()

In [8]:
text

['36824441|t|Very early environmental enrichment protects against apoptosis and improves functional recovery from hypoxic-ischemic brain injury.\n',
 '36824441|a|Appropriate rehabilitation of stroke patients at a very early phase results in favorable outcomes. However, the optimal strategy for very early rehabilitation is at present unclear due to the limited knowledge on the effects of very early initiation of rehabilitation based on voluntary exercise (VE). Environmental enrichment (EE) is a therapeutic paradigm for laboratory animals that involves complex combinations of physical, cognitive, and social stimuli, as well as VE. Few studies delineated the effect of EE on apoptosis in very early stroke in an experimental model. Although a minimal benefit of early rehabilitation in stroke models has been claimed in previous studies, these were based on a forced exercise paradigm. The aim of this study is to determine whether very early exposure to EE can effectively regulate Fas/FasL-med

In [9]:
pubtator_file = f"{PUTATOR_PATH}/{PMID}.txt"
pubtator_process = PubtatorProcessor(pubtator_file, llm)
semantic_blocks = pubtator_process.block
properties_info = pubtator_process.properties_info
pubtator_info = pubtator_process.pubtator_info

2025-03-18 13:38:22,478 - INFO - Processing PubTator file: /home/mindrank/fuli/itext2kg/Data/delirium/36824441.txt


2025-03-18 13:38:53,915 - INFO - HTTP Request: POST http://localhost:8102/v1/chat/completions "HTTP/1.1 200 OK"


In [10]:
properties_info

{'source': 'PMID36824441', 'species': '10090,9606'}

In [11]:
semantic_blocks

["disease - [['disease': 'hypoxic-ischemic brain injury'], ['disease': 'ischemic brain injury'], ['disease': 'stroke'], ['disease': 'hypoxic-ischemic'], ['disease': 'HI)'], ['disease': 'brain injury'], ['disease': 'HI'], ['disease': 'infarcted']]",
 "pathway - [['pathway': 'Fas/FasL-mediated apoptosis'], ['pathway': 'intrinsic apoptotic pathway']]",
 "region - [['region': 'cerebral cortex'], ['region': 'hippocampus']]",
 "gene - [['gene': 'FasL'], ['gene': 'caspase-8'], ['gene': 'caspase-3'], ['gene': 'Bax'], ['gene': 'Bcl-2']]",
 'Title: Very early environmental enrichment protects against apoptosis and improves functional recovery from hypoxic-ischemic brain injury. Abstract: Appropriate rehabilitation of stroke patients at a very early phase results in favorable outcomes. However, the optimal strategy for very early rehabilitation is at present unclear due to the limited knowledge on the effects of very early initiation of rehabilitation based on voluntary exercise (VE). Environment

In [12]:
pubtator_info['abstract'] = {'context':semantic_blocks[-1], 'source':properties_info['source']}

In [13]:
pubtator_info

{'ischemic brain injury': {'label': 'disease', 'unique_id': 'MESH:D001930'},
 'stroke': {'label': 'disease', 'unique_id': 'MESH:D020521'},
 'FasL': {'label': 'gene', 'unique_id': 'Gene ID:14103'},
 'hypoxic-ischemic': {'label': 'disease', 'unique_id': 'MESH:D020925'},
 'HI)': {'label': 'disease', 'unique_id': 'MESH:D020925'},
 'brain injury': {'label': 'disease', 'unique_id': 'MESH:D001930'},
 'HI': {'label': 'disease', 'unique_id': 'MESH:D020925'},
 'infarcted': {'label': 'disease', 'unique_id': 'MESH:D007238'},
 'caspase-8': {'label': 'gene', 'unique_id': 'Gene ID:12370'},
 'caspase-3': {'label': 'gene', 'unique_id': 'Gene ID:12367'},
 'Bax': {'label': 'gene', 'unique_id': 'Gene ID:12028'},
 'Bcl-2': {'label': 'gene', 'unique_id': 'Gene ID:12043'},
 'abstract': {'context': 'Title: Very early environmental enrichment protects against apoptosis and improves functional recovery from hypoxic-ischemic brain injury. Abstract: Appropriate rehabilitation of stroke patients at a very early ph

In [14]:
# -----------------Build the graph-----------------
itext2kg = iText2KG(llm_model = llm, embeddings_model = embeddings)
kg = itext2kg.build_graph(
    sections=[semantic_blocks], 
    source=properties_info,  
    entities_info=pubtator_info,
    ent_threshold=0.9, 
    rel_threshold=0.4,
    )

2025-03-18 13:38:54,168 - INFO - Extracting Entities from the Document
2025-03-18 13:38:54,169 - INFO - entities_info: {'ischemic brain injury': {'label': 'disease', 'unique_id': 'MESH:D001930'}, 'stroke': {'label': 'disease', 'unique_id': 'MESH:D020521'}, 'FasL': {'label': 'gene', 'unique_id': 'Gene ID:14103'}, 'hypoxic-ischemic': {'label': 'disease', 'unique_id': 'MESH:D020925'}, 'HI)': {'label': 'disease', 'unique_id': 'MESH:D020925'}, 'brain injury': {'label': 'disease', 'unique_id': 'MESH:D001930'}, 'HI': {'label': 'disease', 'unique_id': 'MESH:D020925'}, 'infarcted': {'label': 'disease', 'unique_id': 'MESH:D007238'}, 'caspase-8': {'label': 'gene', 'unique_id': 'Gene ID:12370'}, 'caspase-3': {'label': 'gene', 'unique_id': 'Gene ID:12367'}, 'Bax': {'label': 'gene', 'unique_id': 'Gene ID:12028'}, 'Bcl-2': {'label': 'gene', 'unique_id': 'Gene ID:12043'}, 'abstract': {'context': 'Title: Very early environmental enrichment protects against apoptosis and improves functional recovery fro

In [15]:
with open(f'/home/mindrank/fuli/itext2kg/output_kg/delirium/{PMID}.pkl', 'wb') as f:
    pickle.dump(kg, f)

In [16]:
GraphIntegrator(uri=NEO4J_URI, username=NEO4J_USERNAME, password=NEO4J_PASSWORD).visualize_graph(knowledge_graph=kg)

ValueError: Cannot resolve address 435eb4f0.databases.neo4j.io:7687